In [5]:
library(dplyr)
library(Seurat)
library(scop)
library(Signac)
library(ggplot2)
set.seed(4180)
setwd("/")
#########color
cols <- c("#444576", "#4682B4",
 "#AEDEEE","#FFA500",
 "#FFD790","#C65762","#FBDFDE",
 "#F6EFCF","#BCB99F")
pal <- colorRampPalette(cols)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘sp’


The following object is masked from ‘package:IRanges’:

    %over%



Attaching package: ‘SeuratObject’


The following object is masked from ‘package:SummarizedExperiment’:

    Assays


The following object is masked from ‘package:GenomicRanges’:

    intersect


The following object is masked from ‘package:Seqinfo’:

    intersect


The following object is masked from ‘package:IRanges’:

    intersect


The following object is masked from ‘package:S4Vectors’:

    intersect


The following object is masked from ‘package:BiocGenerics’:

    intersect


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘Seurat’


The following object is masked from ‘package:SummarizedExperiment’:

    Assays


Registered S3 method overwritten by 'scop':
  method             from  
  FoldChange.default Seurat

          ⬢          .        ⬡             ⬢     .
  

In [1]:
mHeart <- readRDS("data/mHeart.Rds")

In [13]:
#Supplementary Table 1. Cell type annotation of snRNA-seq and snATAC-seq clusters
library(openxlsx)
meta_data <- mHeart@meta.data
result_df <- meta_data %>%
 dplyr::count(seurat_clusters, celltype, group, name = "count") %>%
 dplyr::arrange(seurat_clusters, celltype, group)
print(head(result_df))
write.xlsx(result_df, "table/Supplementary Table 1. Cell type annotation of snRNA-seq and snATAC-seq clusters.xlsx", rowNames = FALSE)

  seurat_clusters celltype group count
1               0      vCM    EP  1064
2               0      vCM    LP  1257
3               0      vCM    MP  1682
4               0      vCM    NP  2001
5               0      vCM    PP  1857
6               1       FB    EP  1613


In [12]:
mHeart[["activity"]] <- readRDS("data/activity.Rds")[["activity"]]

In [15]:
#Supplementary Table 2. Marker genes for major cardiac cell types and subtypes
rnamarker <- FindAllMarkers(mHeart,min.pct = 0.1,logfc.threshold = 0.5,only.pos = T,assay = 'RNA')
rna_genes <-rnamarker %>%
 dplyr::group_by(cluster) %>%
 dplyr::slice_max(order_by = avg_log2FC, n = 200)
acmarker <- FindAllMarkers(mHeart,min.pct = 0.1,logfc.threshold = 0.5,only.pos = T,assay = 'activity')
acmarker <-acmarker %>%
 dplyr::group_by(cluster) %>%
 dplyr::slice_max(order_by = avg_log2FC, n = 200)
library(openxlsx)
wb <- createWorkbook()
addWorksheet(wb, "snRNA")
addWorksheet(wb, "snATAC")
writeData(wb, sheet = "snRNA", x = rna_genes)
writeData(wb, sheet = "snATAC", x = acmarker)
saveWorkbook(wb, "table/Supplementary Table 2. Marker genes for major cardiac cell types and subtypes.xlsx", overwrite = TRUE)

Calculating cluster vCM



Calculating cluster aCM

Calculating cluster FB

Calculating cluster EC

Calculating cluster EndoCC

Calculating cluster LEC

Calculating cluster SMC

Calculating cluster Pericyte

Calculating cluster Adipocyte

Calculating cluster Neuronal

Calculating cluster T

Calculating cluster B

Calculating cluster Macrophage

Calculating cluster DC

Calculating cluster vCM

Calculating cluster aCM

Calculating cluster FB

Calculating cluster EC

Calculating cluster EndoCC

Calculating cluster LEC

Calculating cluster SMC

Calculating cluster Pericyte

Calculating cluster Adipocyte

Calculating cluster Neuronal

Calculating cluster T

Calculating cluster B

Calculating cluster Macrophage

Calculating cluster DC



In [19]:
#Supplementary Table 3. Differentially expressed genes (DEGs) across reproductive stages
samplegroup <- c("NP", "EP", "MP", "LP", "PP")
mHeart <- RunDEtest(
 srt = mHeart, group.by = "group",
 fc.threshold = 1, only.pos = FALSE, min.pct = 0.1,
 group1 = "NP"
)
DEGs_group <- mHeart@tools$DEtest_custom$AllMarkers_wilcox
for (j in 2:5) {
 mHeart <- RunDEtest(
 srt = mHeart, group.by = "group",
 fc.threshold = 1, only.pos = FALSE, min.pct = 0.1,
 group1 = samplegroup[j], group2 = "NP"
 )
 DEGs_group <- rbind(DEGs_group, mHeart@tools$DEtest_custom$AllMarkers_wilcox)
}
allmarker <- DEGs_group %>%
 dplyr::filter(p_val_adj < 0.05) %>%
 dplyr::filter(pct.1 > 0.1) %>%
 dplyr::filter(avg_log2FC > log2(1.5))
write.xlsx(allmarker, "table/Supplementary Table 3. Differentially expressed genes (DEGs) across reproductive stages.xlsx", rowNames = FALSE)

ℹ [2026-05-20 22:02:13] Data type is log-normalized

ℹ [2026-05-20 22:02:13] Start differential expression test

ℹ [2026-05-20 22:02:13] Find all markers(wilcox) for custom cell groups...

✔ [2026-05-20 22:02:41] Differential expression test completed

ℹ [2026-05-20 22:02:57] Data type is log-normalized

ℹ [2026-05-20 22:02:57] Start differential expression test

ℹ [2026-05-20 22:02:57] Find all markers(wilcox) for custom cell groups...

✔ [2026-05-20 22:03:10] Differential expression test completed

ℹ [2026-05-20 22:03:28] Data type is log-normalized

ℹ [2026-05-20 22:03:28] Start differential expression test

ℹ [2026-05-20 22:03:28] Find all markers(wilcox) for custom cell groups...

✔ [2026-05-20 22:03:38] Differential expression test completed

ℹ [2026-05-20 22:03:56] Data type is log-normalized

ℹ [2026-05-20 22:03:56] Start differential expression test

ℹ [2026-05-20 22:03:56] Find all markers(wilcox) for custom cell groups...

✔ [2026-05-20 22:04:05] Differential expression test

In [20]:
#Supplementary Table 4. GO term enrichment results
library(clusterProfiler)
df_sig <- subset(allmarker)
group <- data.frame(
 gene = df_sig$gene,
 group = df_sig$group1
)
Gene_ID <- bitr(group$gene,
 fromType = "SYMBOL",
 toType = "ENTREZID",
 OrgDb = "org.Mm.eg.db"
)
data <- merge(Gene_ID, group, by.x = "SYMBOL", by.y = "gene")
levels(data$group) <- c("NP", "EP", "MP", "LP", "PP")
data_GO <- compareCluster(
 ENTREZID ~ group,
 data = data,
 fun = "enrichGO",
 OrgDb = "org.Mm.eg.db",
 ont = "BP",
 pAdjustMethod = "BH",
 pvalueCutoff = 0.05,
 qvalueCutoff = 0.05
)
res <- data_GO@compareClusterResult
for (j in 1:dim(res)) {
 arr <- unlist(strsplit(as.character(res[j, "geneID"]), split = "/"))
 gene_names <- paste(unique(data$SYMBOL[data$ENTREZID %in% arr]), collapse = "/")
 res[j, "geneID"] <- gene_names
}
write.xlsx(res, "table/Supplementary Table 4. GO term enrichment results.xlsx")



clusterProfiler v4.18.4 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

G Yu. Thirteen years of clusterProfiler. The Innovation. 2024,
5(6):100722


Attaching package: ‘clusterProfiler’


The following object is masked from ‘package:stats’:

    filter




'select()' returned 1:1 mapping between keys and columns

Warning message in bitr(group$gene, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = "org.Mm.eg.db"):
“2.99% of input gene IDs are fail to map...”
Warning message in 1:dim(res):
“numerical expression has 2 elements: only the first used”


In [7]:
head(figR.d)

,DORC,Motif,Enrichment.Z,Enrichment.P,Enrichment.log10P,Corr,Corr.Z,Corr.P,Corr.log10P,Score
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1700025G04Rik,Ahctf1,-0.78066855,0.4349975,-0.36151327,-0.0176332265,-0.1835387,0.8543754,-0.06835128,0.00000000
2,1700025G04Rik,Ahr,-0.05834459,0.9534742,-0.02069108,-0.0023125449,0.3650391,0.7150822,0.14564404,0.00000000
3,1700025G04Rik,Aire,0.39874287,0.6900827,0.16109888,0.0076549818,0.7219400,0.4703314,0.32759604,0.07787346
4,1700025G04Rik,Alx4,0.28528304,0.7754273,0.11045891,0.0002149095,0.4555381,0.6487222,0.18794125,0.03568728
5,1700025G04Rik,Ar,1.47031448,0.1414766,0.84931538,-0.0019661898,0.3774408,0.7058460,0.15129003,-0.12641093
6,1700025G04Rik,Arid2,-1.28187027,0.1998882,-0.69921294,-0.0446957147,-1.1525479,0.2490960,-0.60363321,0.00000000


In [9]:
#Supplementary Table 5. TF–target interactions from eGRN networks
library(FigR)
library(openxlsx)
figR.d <- readRDS("data/vCM2_figR.rds")
eGRN <- figR.d %>%
 dplyr::filter(Score > 0.1 | Score < -0.1)
write.xlsx(eGRN, "table/Supplementary Table 5. TF–target interactions from eGRN networks.xlsx", rowNames = FALSE)